# 02 · Limpieza y verificación de datos

**Fase 2 del E3.** Partimos de los Parquet generados en `01_ingesta` y dejamos los datos listos para el análisis exploratorio y el modelado.

| Problema | Decisión |
|---|---|
| Valores nulos (≤ 0,9 % por columna) | Lectura anterior del camión → lectura siguiente → mediana de train |
| Variables acumuladas que bajan | Corregir con un desfase y marcar la lectura (`flag_reinicio`) |
| Saltos atípicos en contadores | Solo marcar (`flag_salto`); pueden ser la señal del fallo |
| Camiones con pocas lecturas | Mantener; también existen en validación y test |
| Columnas casi constantes | Mantener; se decide en la ingeniería de características |
| Categorías raras en especificaciones | Agrupar en `Otras` (< 50 camiones en train) |

Toda la lógica está en `src/cleaning.py` (clase `Cleaner`): se **ajusta solo con train** (medianas, umbrales, categorías) y se aplica igual a train, validación, test y a la API de BentoML, para no filtrar información y que todas las partes del proyecto limpien los datos de la misma forma.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import data
from src.cleaning import (
    SENSOR_GROUPS, Cleaner, build_clean, growth_rates, load_clean,
    monotonic_violations, null_pattern, sort_readouts,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

Para el análisis cargamos las lecturas en `float32` (la mitad de memoria). La limpieza final (`build_clean`, sección 7) trabaja en `float64` y carga los conjuntos de uno en uno.

In [ ]:
splits = {s: data.load_split(s, float32=True) for s in data.SPLITS}
train_r, train_s, train_tte = splits["train"]
print({s: r.shape for s, (r, _, _) in splits.items()})

## 1. Valores nulos

### 1.1 Patrón: ¿faltan valores sueltos o sensores enteros?

In [ ]:
patterns = pd.concat({s: null_pattern(r).set_index("sensor") for s, (r, _, _) in splits.items()}, axis=1)
patterns.xs("% lecturas", axis=1, level=1).join(
    patterns[("train", "lecturas nulo parcial")].rename("train: nulos parciales")
)

In [ ]:
pct = patterns.xs("% lecturas", axis=1, level=1).sort_values("train", ascending=True)
ax = pct.plot.barh(figsize=(7, 5), width=0.8)
ax.set_xlabel("% de lecturas con el sensor nulo")
ax.set_title("Nulos por sensor y conjunto")
plt.tight_layout()

**Conclusión.** Ningún sensor tiene nulos parciales: cuando falta un sensor, faltan **todas** sus columnas a la vez. Es decir, en esa lectura la ECU no registró la señal o no se pudo descargar; no son errores de celdas sueltas. Los sensores más afectados son 291, 459, 100, 370 y 427 (< 1 % de las lecturas) y el patrón es el mismo en los tres conjuntos.

### 1.2 ¿En qué lecturas aparecen?

In [ ]:
def null_position(r):
    na = r[data.SENSOR_COLUMNS].isna().any(axis=1)
    v = r["vehicle_id"].to_numpy()
    first = np.r_[True, v[1:] != v[:-1]]
    last = np.r_[v[1:] != v[:-1], True]
    per_vehicle = r[data.SENSOR_COLUMNS].isna().groupby(r["vehicle_id"]).all().any(axis=1)
    return {
        "lecturas con nulos": int(na.sum()),
        "en la primera lectura del camión": int((na & first).sum()),
        "en la última lectura del camión": int((na & last).sum()),
        "camiones con algún sensor siempre nulo": int(per_vehicle.sum()),
    }

pd.DataFrame({s: null_position(r) for s, (r, _, _) in splits.items()})

**Decisión de imputación.** Las variables son acumuladas, así que el mejor valor para una lectura que falta es el **último valor conocido del mismo camión** (*forward fill*): supone que el sensor no avanzó en ese intervalo, nunca inventa un valor mayor del que se observó. Si el nulo está en la primera lectura del camión se usa la siguiente (*backward fill*), y si el camión no tiene ese sensor en ninguna lectura, la **mediana de train**.

Importante: en validación y test la **última lectura es la que se clasifica**, y algunas tienen nulos. Con el *forward fill* esas lecturas usan el último valor conocido, igual que se haría en producción.

Además se guarda `n_sensores_imputados` por lectura, por si el número de sensores que faltan aporta información al modelo.

## 2. Variables acumuladas que bajan

En `01_ingesta` encontramos 7 descensos en el contador `171_0` comparando cada lectura con la anterior. Aquí afinamos la comprobación: comparamos con el **último valor válido**, para detectar también los descensos que quedan escondidos detrás de un hueco de nulos, y la extendemos a las columnas de los histogramas (que también son acumuladas).

In [ ]:
violations = {s: monotonic_violations(r) for s, (r, _, _) in splits.items()}

pd.DataFrame({
    s: {
        "descensos (valores)": len(v),
        "camiones afectados": v["vehicle_id"].nunique(),
        "tras un hueco de nulos": int(v["tras hueco de nulos"].sum()),
        "sensores": ", ".join(sorted(v["sensor"].unique())),
    }
    for s, v in violations.items()
})

In [ ]:
v = violations["train"]
v.groupby(["sensor", "tras hueco de nulos"]).agg(
    descensos=("descenso", "size"),
    camiones=("vehicle_id", "nunique"),
    descenso_mediano=("descenso", "median"),
    descenso_relativo_mediano=("descenso relativo", "median"),
)

### 2.1 Casos sin hueco de nulos: los 7 descensos de `171_0`

In [ ]:
direct = v[~v["tras hueco de nulos"]]
direct[["vehicle_id", "time_step", "variable", "valor anterior", "descenso"]]

Los 7 descensos son de exactamente **−15**, que es la resolución del contador (sus valores van de 15 en 15): son errores de redondeo o de registro de una sola unidad, no reinicios.

### 2.2 Casos tras un hueco de nulos

In [ ]:
def plot_vehicle(r, vehicle_id, column, ax=None):
    x = r[r["vehicle_id"] == vehicle_id]
    ax = ax or plt.gca()
    ax.plot(x["time_step"], x[column], "o-", ms=3)
    nulls = x[x[column].isna()]["time_step"]
    for t in nulls:
        ax.axvline(t, color="lightgray", lw=4, zorder=0)
    ax.set_title(f"camión {vehicle_id} · {column}")
    ax.set_xlabel("time_step")

gap = v[v["tras hueco de nulos"]]
examples = gap.sort_values("descenso relativo", ascending=False).drop_duplicates("vehicle_id").head(3)
fig, axes = plt.subplots(1, len(examples), figsize=(13, 3.5))
for ax, (_, row) in zip(np.atleast_1d(axes), examples.iterrows()):
    plot_vehicle(train_r, row["vehicle_id"], row["variable"], ax)
plt.tight_layout()

Las bandas grises son lecturas en las que el sensor no registró nada. Al volver a registrar, el valor es menor que antes del hueco: el sensor (o su contador en la ECU) **se reinició** mientras no había datos, y a partir de ahí vuelve a acumular normalmente.

**Decisión: corrección por desfase.** En cada descenso se suma la diferencia con el último valor válido a esa lectura y a todas las siguientes del camión. Así la serie sigue creciendo y se conservan los incrementos reales posteriores, que son los que usaremos para calcular ritmos y tendencias. La alternativa de aplicar el máximo acumulado dejaría la serie plana durante muchas lecturas cuando el reinicio es grande, y eso sí distorsionaría las tendencias. Las lecturas corregidas se marcan con `flag_reinicio`.

## 3. Saltos atípicos

Los contadores no pueden bajar, pero sí pueden crecer de forma anómalamente rápida entre dos lecturas. Medimos el **ritmo de crecimiento** (incremento / Δ`time_step`) de cada contador. Como es muy asimétrico (cola larga a la derecha), el umbral IQR se calcula sobre `log(1 + ritmo)`: un salto es atípico si supera `Q3 + 3·IQR` de train.

In [ ]:
cleaner_tmp = Cleaner().fit(train_r, train_s)
rates = growth_rates(sort_readouts(train_r), data.COUNTERS)
log_rates = np.log1p(rates.clip(lower=0))

fig, axes = plt.subplots(2, 4, figsize=(14, 5.5))
for ax, c in zip(axes.ravel(), data.COUNTERS):
    ax.hist(log_rates[c].dropna(), bins=80, color="tab:blue", alpha=0.8)
    ax.axvline(cleaner_tmp.rate_limits_[c], color="tab:red", ls="--", label="umbral")
    ax.set_title(c); ax.set_yscale("log")
axes[0, 0].legend()
fig.suptitle("log(1 + ritmo de crecimiento) por contador en train")
plt.tight_layout()

In [ ]:
pd.DataFrame({
    "umbral log(1+ritmo)": cleaner_tmp.rate_limits_,
    "lecturas por encima (train)": log_rates.gt(pd.Series(cleaner_tmp.rate_limits_), axis=1).sum(),
})

**Decisión: marcar, no corregir.** Son muy pocas lecturas y no hay forma de saber si son errores o un uso realmente intenso; en mantenimiento predictivo un comportamiento extremo puede ser justo la señal previa a una avería. Se guardan con `flag_salto` para analizarlas en el EDA. No hay valores negativos en ninguna variable.

In [ ]:
print("Valores negativos:", int((train_r[data.SENSOR_COLUMNS] < 0).sum().sum()))

## 4. Estructura temporal

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
dt = train_r.sort_values(["vehicle_id", "time_step"]).groupby("vehicle_id")["time_step"].diff()
axes[0].hist(dt.dropna(), bins=100, range=(0, 30)); axes[0].set_title("Δ time_step entre lecturas (train)")
n_read = {s: r.groupby("vehicle_id").size() for s, (r, _, _) in splits.items()}
for s, n in n_read.items():
    axes[1].hist(n, bins=60, range=(0, 200), histtype="step", lw=1.5, label=s, density=True)
axes[1].set_title("Lecturas por camión"); axes[1].legend()
axes[2].hist(train_tte["length_of_study_time_step"], bins=60)
axes[2].set_title("Duración de la observación (train)")
plt.tight_layout()

print(dt.describe().round(2).to_dict())

In [ ]:
pd.DataFrame({
    s: {f"< {k} lecturas": f"{(n < k).sum()} ({(n < k).mean():.1%})" for k in (3, 5, 10)}
    for s, n in n_read.items()
})

**Decisión.** El intervalo entre lecturas es irregular (mediana ≈ 4 `time_step`, con huecos de cientos), así que en la ingeniería de características los ritmos se calcularán **por unidad de `time_step`** y no por número de lecturas. Los camiones con pocas lecturas no se eliminan: validación y test tienen la misma proporción y el modelo tendrá que predecir para ellos; sus características de tendencia se calcularán con las lecturas que haya.

## 5. Columnas casi constantes

In [ ]:
zero_share = (train_r[data.SENSOR_COLUMNS] == 0).mean().sort_values(ascending=False)
zero_share[zero_share > 0.8].rename("% lecturas = 0").to_frame()

`272_8` y `272_9` (los dos últimos intervalos del histograma 272) valen 0 en más del 95 % de las lecturas. **No se eliminan aquí**: precisamente los intervalos extremos pueden indicar operación exigente (hipótesis H2). Se decidirá en la ingeniería de características con criterios de varianza e importancia.

## 6. Especificaciones: categorías raras

In [ ]:
spec_summary = pd.DataFrame({
    c: {
        "categorías": train_s[c].nunique(),
        "con < 50 camiones": int((train_s[c].value_counts() < 50).sum()),
        "camiones en esas categorías": int(train_s[c].value_counts()[lambda x: x < 50].sum()),
    }
    for c in data.SPEC_COLUMNS
}).T
spec_summary

`Spec_1`, `Spec_2` y `Spec_6` tienen muchas categorías con muy pocos camiones (alguna con uno solo). Con *one-hot* generarían columnas casi vacías y estimaciones poco fiables de la tasa de fallo. **Decisión:** las categorías con menos de 50 camiones en train, y las que aparezcan en validación/test pero no en train, se agrupan en `Otras`.

## 7. Aplicación de la limpieza

`build_clean()` ajusta el `Cleaner` con train, lo guarda en `models/cleaner.json` (lo usará la API de BentoML) y limpia los tres conjuntos en `data/processed/clean/`.

In [ ]:
# liberamos memoria: build_clean vuelve a cargar cada conjunto
del splits, train_r, rates, log_rates
import gc; gc.collect()

report = build_clean()
report

## 8. Verificación tras la limpieza

In [ ]:
rows = {}
for s in data.SPLITS:
    r, sp, lb = load_clean(s)
    rows[s] = {
        "lecturas": len(r),
        "camiones": r["vehicle_id"].nunique(),
        "nulos": int(r[data.SENSOR_COLUMNS].isna().sum().sum()),
        "descensos": len(monotonic_violations(r)),
        "lecturas con flag_reinicio": int(r["flag_reinicio"].sum()),
        "lecturas con flag_salto": int(r["flag_salto"].sum()),
        "lecturas con sensores imputados": int((r["n_sensores_imputados"] > 0).sum()),
        "categorías Spec_1": sp["Spec_1"].nunique(),
    }
check = pd.DataFrame(rows)
check

In [ ]:
assert (check.loc["nulos"] == 0).all(), "Quedan nulos"
assert (check.loc["descensos"] == 0).all(), "Quedan descensos"
assert (check.loc["lecturas"].values == [1_122_452, 196_227, 198_140]).all(), "Se han perdido lecturas"
print("Limpieza verificada: sin nulos, sin descensos y sin lecturas perdidas.")

## 9. Resumen de decisiones

| Paso | Qué se ha hecho | Por qué |
|---|---|---|
| Nulos | `ffill` por camión → `bfill` → mediana de train; `n_sensores_imputados` | Variables acumuladas: el último valor conocido es la estimación más prudente y la única disponible en producción |
| Descensos | Corrección por desfase + `flag_reinicio` | Son reinicios del sensor/ECU (casi todos tras huecos de nulos) o errores de una unidad; el desfase conserva los incrementos reales |
| Saltos | Umbral `Q3 + 3·IQR` sobre `log(1+ritmo)`; solo `flag_salto` | Pueden ser señal de fallo; no hay forma de distinguirlos de errores |
| Pocas lecturas | Se mantienen | Existen igual en validación y test |
| Columnas casi constantes | Se mantienen | Se decide con criterios de importancia en la Fase 4 |
| Especificaciones | Categorías < 50 camiones → `Otras` | Evita columnas *one-hot* casi vacías y estimaciones inestables |
| Fuga de datos | Todo se ajusta solo con train | Validación y test se tratan como datos nuevos, igual que en producción |

**Siguiente paso:** `03_eda.ipynb`, sobre los datos de `data/processed/clean/`.